# Regulatory programs and cross-modality transfer in C. elegans

This notebook follows the biological workflow from real input data to a trained model, a newly selected panel, and manuscript-matched biological analyses. It does not read packaged aggregate results as tutorial inputs. [Open the editable source notebook on GitHub](https://github.com/fym0503/SMITH/blob/main/docs/source/tutorials/notebooks/regulatory_section/02_SMITH_Regulatory_Activity_source.ipynb).

## Biological question

Which compact set of regulatory features is sufficient to preserve C. elegans cell identity and developmental progression? The TF and miRNA assays represent regulatory activity rather than a generic feature-selection benchmark: a useful panel should retain discrete lineage labels and the continuous developmental-time signal in held-out cells.

**How to read the endpoint:** Cell-type accuracy asks whether the panel preserves discrete lineage information, while developmental-time correlation asks whether it preserves the ordered embryonic trajectory. The paper-specific analyses then ask three biological follow-up questions: does the panel retain annotated developmental modules, can selected targets reconstruct TF co-activity in muscle, neuron, pharynx and skin, and does a panel selected from scRNA-seq retain cell identity when its genes are evaluated on held-out TF-activity lineages?

## Step 0: Download the real input data

Download the versioned Zenodo archive and verify its checksums before training:

```bash
python scripts/download_tutorial_data.py \
  --case 02_regulatory_activity \
  --data-root data/tutorials
```

The notebook is pre-executed for documentation. Read the Docs does not download large data or train SMITH during documentation builds.

The developmental-module and TF-pair tables in the archive are normalized from the source atlas Supplementary Table 5. Their preparation can be audited independently with:

```bash
python scripts/prepare_elegans_atlas_annotations.py --data-root data/tutorials
```

## Configuration

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "reproducibility").exists():
            return candidate
    raise RuntimeError("Run this notebook inside a SMITH repository checkout.")

ROOT = find_repository(Path.cwd().resolve())
DATA_ROOT = Path(os.environ.get("SMITH_TUTORIAL_DATA", "data/tutorials")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("SMITH_TUTORIAL_OUTPUT", "outputs/tutorials")).expanduser().resolve()
CASE_OUTPUT = OUTPUT_ROOT / 'regulatory'
EPOCHS = int(os.environ.get("SMITH_TUTORIAL_EPOCHS", 30))
DEVICE = os.environ.get("SMITH_TUTORIAL_DEVICE", 'cuda:0')
REUSE_EXISTING = os.environ.get("SMITH_TUTORIAL_REUSE_EXISTING", "0") == "1"
WORKFLOW_ENV = os.environ.copy()
WORKFLOW_ENV["PYTHONPATH"] = os.pathsep.join(
    [str(ROOT), str(ROOT / "src"), WORKFLOW_ENV.get("PYTHONPATH", "")]
)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


from reproducibility.workflows.figure_style import configure
from reproducibility.workflows.regulatory_activity.plot_figure3 import (
    PANEL_SPECS, _draw_bar_panel, _draw_coactivity, _draw_module_coverage,
    _draw_module_schematic, _draw_transfer, _plot_tf_correlation,
)
configure()


## Step 1: Inspect the biological input data

The train/test H5AD files contain lineage-aware TF or miRNA activity, cell-type labels, and absolute developmental time. The supplementary module and TF-pair annotations define the developmental programs and regulator relationships used in the manuscript analyses, while the scRNA H5AD supplies the independent reference for RNA-to-TF transfer. Training cells learn the activity representation; held-out cells test whether selected regulators still recover identity, age, and regulatory structure.

In [ ]:
inputs = ['regulatory_activity/elegans/splits/elegans_tf/split_1/train.h5ad', 'regulatory_activity/elegans/splits/elegans_tf/split_1/test.h5ad', 'regulatory_activity/elegans/splits/elegans_mirna/split_1/train.h5ad', 'regulatory_activity/elegans/splits/elegans_mirna/split_1/test.h5ad', 'regulatory_activity/elegans/annotations/tf_spatiotemporal_modules.tsv', 'regulatory_activity/elegans/annotations/tf_regulatory_pairs.tsv', 'regulatory_activity/elegans/reference/elegans_scrna.h5ad']
input_checksums = {}
for relative in inputs:
    path = DATA_ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Run scripts/download_tutorial_data.py first.")
    input_checksums[relative] = sha256_file(path)


## Step 2: Train SMITH and select a panel

SMITH is trained on the training H5AD with reconstruction, cell-type classification, and developmental-time objectives. Its learned gene ranking is then truncated to the manuscript panel sizes; no packaged aggregate ranking is used.

The command below starts from the H5AD inputs above and writes a fresh model ranking, panel, evaluation, and run manifest.

In [ ]:
command = [sys.executable, str(ROOT / 'reproducibility/workflows/regulatory_activity/run_tutorial.py'), "--data-root", str(DATA_ROOT), "--output-dir", str(CASE_OUTPUT), "--device", DEVICE, "--epochs", str(EPOCHS)] + ['--datasets', 'elegans_tf,elegans_mirna', '--splits', 'split_1', '--methods', 'SMITH', '--seeds', '1', '--max-cells', '3000', '--paper-analyses', '--module-file', 'regulatory_activity/elegans/annotations/tf_spatiotemporal_modules.tsv', '--regulatory-pair-file', 'regulatory_activity/elegans/annotations/tf_regulatory_pairs.tsv', '--scrna-file', 'regulatory_activity/elegans/reference/elegans_scrna.h5ad']
if not REUSE_EXISTING:
    command.append("--force")
completed = subprocess.run(
    command, cwd=ROOT, env=WORKFLOW_ENV, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
if completed.returncode:
    print(completed.stdout)
    raise subprocess.CalledProcessError(completed.returncode, command)
manifest = json.loads((CASE_OUTPUT / "run_manifest.json").read_text())
if not manifest.get("training_runs"):
    raise RuntimeError("The workflow did not record any SMITH training runs.")
for relative in ['figure_data/figure3_c_f_summary.tsv', 'figure_data/figure3_c_f_paired_tests.tsv', 'figure_data/figure3_h_module_miss_rate.tsv', 'figure_data/figure3_i_coactivity.tsv', 'figure_data/figure3_j_tf_scrna_correlation.tsv', 'figure_data/figure3_k_transfer.tsv']:
    if not (CASE_OUTPUT / relative).is_file():
        raise FileNotFoundError(CASE_OUTPUT / relative)


## Preserving cell identity and developmental progression

### TF activity retains lineage identity

The selected TF panel is evaluated on held-out lineages with the same 5-nearest-neighbour classifier used for the manuscript comparison. Higher accuracy means that a compact regulatory panel still separates cell identities.

In [ ]:
import warnings
from reproducibility.workflows.regulatory_activity.evaluate_outputs import evaluate

panel_file = CASE_OUTPUT / "runs/elegans_tf/split_1/seed_1/panel_32/panels/SMITH_top32.tsv"
recheck_dir = CASE_OUTPUT / "notebook_recheck/elegans_tf"
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
    _ = evaluate(
        DATA_ROOT / "regulatory_activity/elegans/splits/elegans_tf/split_1/train.h5ad",
        DATA_ROOT / "regulatory_activity/elegans/splits/elegans_tf/split_1/test.h5ad",
        panel_file, recheck_dir, 32, neighbors=5,
    )
tf_values = pd.read_csv(CASE_OUTPUT / "figure_data/figure3_c_f_values.tsv", sep="	")
figure, axis = plt.subplots(figsize=(2.35, 2.10), facecolor="white")
_draw_bar_panel(axis, tf_values, PANEL_SPECS["c"])
figure.text(0.015, 0.985, "c", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.25, right=0.97, bottom=0.22, top=0.86)
display(figure)
plt.close(figure)

### TF activity retains developmental order

Developmental time is predicted independently from the same held-out TF cells. Pearson correlation measures whether the panel preserves the continuous temporal trajectory rather than only discrete labels.

In [ ]:
from reproducibility.workflows.regulatory_activity.analysis import write_statistical_analysis

_ = write_statistical_analysis(
    CASE_OUTPUT / "figure_data/figure3_c_f_values.tsv",
    CASE_OUTPUT / "figure_data",
)
tf_values = pd.read_csv(CASE_OUTPUT / "figure_data/figure3_c_f_values.tsv", sep="	")
figure, axis = plt.subplots(figsize=(2.35, 2.10), facecolor="white")
_draw_bar_panel(axis, tf_values, PANEL_SPECS["d"])
figure.text(0.015, 0.985, "d", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.25, right=0.97, bottom=0.22, top=0.86)
display(figure)
plt.close(figure)

### miRNA activity retains lineage identity

The same held-out analysis is repeated in the miRNA activity space, where smaller panel sizes test whether post-transcriptional regulation contains sufficient lineage information.

In [ ]:
import warnings
from reproducibility.workflows.regulatory_activity.evaluate_outputs import evaluate

panel_file = CASE_OUTPUT / "runs/elegans_mirna/split_1/seed_1/panel_32/panels/SMITH_top32.tsv"
recheck_dir = CASE_OUTPUT / "notebook_recheck/elegans_mirna"
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")
    _ = evaluate(
        DATA_ROOT / "regulatory_activity/elegans/splits/elegans_mirna/split_1/train.h5ad",
        DATA_ROOT / "regulatory_activity/elegans/splits/elegans_mirna/split_1/test.h5ad",
        panel_file, recheck_dir, 32, neighbors=5,
    )
mirna_values = pd.read_csv(CASE_OUTPUT / "figure_data/figure3_c_f_values.tsv", sep="	")
figure, axis = plt.subplots(figsize=(2.35, 2.10), facecolor="white")
_draw_bar_panel(axis, mirna_values, PANEL_SPECS["e"])
figure.text(0.015, 0.985, "e", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.25, right=0.97, bottom=0.22, top=0.86)
display(figure)
plt.close(figure)

### miRNA activity retains developmental order

The temporal endpoint asks whether the compact miRNA panel follows embryonic progression across held-out cells.

In [ ]:
mirna_values = pd.read_csv(CASE_OUTPUT / "figure_data/figure3_c_f_values.tsv", sep="\t")
figure, axis = plt.subplots(figsize=(2.35, 2.10), facecolor="white")
_draw_bar_panel(axis, mirna_values, PANEL_SPECS["f"])
figure.text(0.015, 0.985, "f", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.25, right=0.97, bottom=0.22, top=0.86)
display(figure)
plt.close(figure)

## Developmental regulatory programs

### Spatiotemporal TF modules

The atlas annotation organizes regulators by tissue system and temporal module. This establishes the biological programs whose coverage is tested in the next analysis.

In [ ]:
modules = pd.read_csv(DATA_ROOT / "regulatory_activity/elegans/annotations/tf_spatiotemporal_modules.tsv", sep="\t")
figure, axis = plt.subplots(figsize=(2.55, 2.25), facecolor="white")
_draw_module_schematic(axis, modules)
figure.text(0.015, 0.985, "g", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.16, right=0.97, bottom=0.23, top=0.85)
display(figure)
plt.close(figure)

### Coverage of developmental modules

For each newly selected panel, the miss rate is the fraction of annotated modules with no selected TF. A lower value therefore indicates broader coverage of known developmental programs.

In [ ]:
module_coverage = pd.read_csv(CASE_OUTPUT / "figure_data/figure3_h_module_miss_rate.tsv", sep="\t")
figure, axis = plt.subplots(figsize=(2.55, 2.25), facecolor="white")
_draw_module_coverage(axis, module_coverage)
figure.text(0.015, 0.985, "h", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.16, right=0.97, bottom=0.23, top=0.85)
display(figure)
plt.close(figure)

## Regulatory reconstruction and modality transfer

### Reconstruction of TF co-activity

The trained reconstruction head predicts held-out TF activity from the selected regulators. Agreement is measured within muscle, neuronal, pharyngeal and skin lineages using atlas-defined TF pairs.

In [ ]:
coactivity = pd.read_csv(CASE_OUTPUT / "figure_data/figure3_i_coactivity.tsv", sep="\t")
lineages = ["muscle", "neuron", "pharynx", "skin"]
methods = [name for name in ("SMITH", "PERSIST") if name in set(coactivity["method"])]
colors = {"SMITH": "#2f75b5", "PERSIST": "#f2b134"}
x = np.arange(len(lineages))
bar_width = 0.72 / len(methods)

figure, axis = plt.subplots(figsize=(2.55, 2.25), facecolor="white")
for method_index, method in enumerate(methods):
    means, errors = [], []
    for lineage in lineages:
        values = coactivity.loc[
            (coactivity["method"] == method) & (coactivity["lineage"] == lineage),
            "pearson",
        ].dropna()
        means.append(values.mean())
        errors.append(values.sem() if len(values) > 1 else 0.0)
    positions = x - 0.36 + bar_width / 2 + method_index * bar_width
    axis.bar(
        positions, means, bar_width, yerr=errors, capsize=1.5,
        color=colors[method], edgecolor="black", linewidth=0.4, label=method,
    )

axis.set(title="TF co-activity reconstruction", ylabel="Pearson agreement")
axis.set_xticks(x, [name.title() for name in lineages], rotation=25, ha="right")
axis.set_ylim(-1, 1)
axis.legend(frameon=False, fontsize=5.5)
figure.text(0.015, 0.985, "i", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.16, right=0.97, bottom=0.23, top=0.85)
display(figure)
plt.close(figure)

### Conservation of regulatory structure across modalities

Shared TFs are aggregated over matched lineages in scRNA-seq and TF-activity data, then biclustered using the TF-activity correlation matrix. Similar block structure indicates that inferred regulatory activity preserves transcriptomic organization.

In [ ]:
correlation = pd.read_csv(CASE_OUTPUT / "figure_data/figure3_j_tf_scrna_correlation.tsv", sep="\t")
figure = _plot_tf_correlation(correlation, CASE_OUTPUT / "figure_data/figure3_j_tf_scrna_correlation.tsv")
figure.text(0.008, 0.985, "j", ha="left", va="top", fontsize=10, weight="bold")
display(figure)
plt.close(figure)

### Transfer from scRNA-seq into TF activity

Panels selected from scRNA-seq are evaluated on held-out TF-activity lineages. The comparison with TF-selected panels tests whether gene choice transfers across molecular representations without losing cell identity.

In [ ]:
transfer = pd.read_csv(CASE_OUTPUT / "figure_data/figure3_k_transfer.tsv", sep="\t")
figure, axis = plt.subplots(figsize=(2.55, 2.25), facecolor="white")
_draw_transfer(axis, transfer)
figure.text(0.015, 0.985, "k", ha="left", va="top", fontsize=10, weight="bold")
figure.subplots_adjust(left=0.16, right=0.97, bottom=0.23, top=0.85)
display(figure)
plt.close(figure)

## Full manuscript command

Append the following paper-scale arguments to the workflow command:

```text
--splits split_1,split_2,split_3,split_4,split_5 --methods SMITH,PERSIST-class,PERSIST,ActiveSVM,scGIST,scGeneFit,Spapros --baseline-root external/SMITH_baselines/GPS_tools-main/baselines --baseline-python PERSIST=/opt/envs/persist/bin/python --baseline-python PERSIST-class=/opt/envs/persist/bin/python --baseline-python scGIST=/opt/envs/scgist/bin/python --epochs 200
```

This executed page uses one real TF split, one real miRNA split and one scRNA-to-TF transfer split. The paper command above regenerates Figure 3c-k with all five lineage-aware splits and manuscript baselines. The module, TF-pair and scRNA inputs are versioned biological inputs; the workflow stops with an explicit error if they are absent. This quick hosted run is intentionally smaller than the paper-scale comparison, but it starts from the same real inputs and executes the same prediction and analysis functions.